# BTC Spot–Perpetual Basis: Single-Day Causal State-Space Research

This workbench studies one settled UTC day of real Binance `BTCUSDT` spot and USD-M perpetual `aggTrades` data.

## Fixed scope contract

This repository is **intentionally a single-day methodological research project**. Its purpose is to demonstrate causal signal construction, chronological holdout discipline, timestamp-correct proxy economics, reproducibility and falsification on one day.

- Extending this notebook to multiple days is **not a missing task or backlog item for this repository**.
- Multi-day regime stability, rolling refits and cross-date validation belong to a **separate follow-on project**.
- This repository must therefore be judged as a one-day research workbench, not criticised for deliberately excluding a different project.

The notebook does not claim executable P&L. `aggTrades` contains market trade prints, not historical bid/ask depth, queue position or the researcher's counterfactual fills.


## 1. Research map

The day is divided chronologically into four non-overlapping intervals:

1. **Training:** fit the state-space model and the AR(1)/OU baseline.
2. **Threshold development:** run an explicitly exploratory threshold sensitivity sweep.
3. **Purge:** prevent an episode initiated in development from completing in the final holdout.
4. **Final holdout:** evaluate the predeclared signal configuration and causal baselines once.

The one-sided state-space filter continues through all intervals because an online filter naturally carries its state forward. Model parameters are never refitted after the training cutoff.


In [ ]:
from __future__ import annotations

import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from research_backtest import (
    ProxyBacktestConfig,
    ProxyBacktestResult,
    run_proxy_backtest_detailed,
)
from research_baselines import (
    ar1_ou_signal_z,
    causal_rolling_z,
    fit_ar1_ou_baseline,
)
from research_data import (
    fetch_archive,
    installed_versions,
    write_research_manifest,
)
from research_validation import (
    benjamini_hochberg,
    circular_block_bootstrap_mean_ci,
    circular_block_bootstrap_mean_test,
    make_intraday_research_split,
)
from state_space import filter_state_space, fit_state_space

plt.rcParams.update({
    "figure.figsize": (12, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)


## 2. Configuration

All primary choices are declared before the data is analysed. The final-holdout strategy uses `ENTRY_Z=2.0`; the development sweep does not select a replacement threshold.

The purge duration is tied to the maximum holding and fill-delay assumptions, rather than chosen as an arbitrary number.


In [ ]:
SYMBOL = "BTCUSDT"
DATE = "2026-07-20"
BAR_SECONDS = 1
CACHE_DIR = Path("./data")
OUTPUT_DIR = Path("./artifacts")

MAX_STALE_SECONDS = 2.0
TRAIN_DURATION = "60000s"
DEVELOPMENT_DURATION = "8000s"
MINIMUM_HOLDOUT_DURATION = "4h"

ENTRY_Z = 2.0
EXIT_Z = 0.25
STOP_Z = 4.0
MAX_HOLD_SECONDS = 60.0
COOLDOWN_SECONDS = 5.0
EXECUTION_DELAY_SECONDS = 1.0
MAX_FILL_DELAY_SECONDS = 2.0

FEE_BP_PER_FILL = 2.0
SLIPPAGE_BP_PER_FILL = 0.10
ALLOW_SHORT_SPOT = False
RUN_DIAGNOSTIC_BACKTEST = True

ROLLING_WINDOW_SECONDS = 300
ROLLING_MIN_PERIODS = 120
THRESHOLD_GRID = np.arange(0.75, 4.01, 0.25)
BOOTSTRAP_REPLICATIONS = 2_000

PURGE_SECONDS = (
    MAX_HOLD_SECONDS
    + EXECUTION_DELAY_SECONDS
    + MAX_FILL_DELAY_SECONDS
)
ROUND_TRIP_COST_BP = 4.0 * (FEE_BP_PER_FILL + SLIPPAGE_BP_PER_FILL)

PERP_URL = "https://data.binance.vision/data/futures/um/daily/aggTrades"
SPOT_URL = "https://data.binance.vision/data/spot/daily/aggTrades"


## 3. Checksummed data acquisition

Raw ZIP archives are retained under `data/raw/` and SHA-256 checksummed. Parsed caches include the archive digest in their filename, so a changed archive cannot silently reuse a stale parsed object.

The full regular one-second clock is preserved. A stale or missing observation becomes `NaN`; rows are not deleted because row deletion would compress elapsed time and corrupt both the state equation and holding-period logic.


In [ ]:
PERP_COLUMNS = [
    "agg_trade_id", "price", "quantity", "first_trade_id",
    "last_trade_id", "transact_time", "is_buyer_maker",
]
SPOT_COLUMNS = PERP_COLUMNS + ["is_best_match"]


def _timestamp_unit(values: pd.Series) -> str:
    finite = pd.to_numeric(values, errors="coerce").dropna()
    if finite.empty:
        raise ValueError("No valid timestamps")
    return "us" if float(finite.median()) > 1e14 else "ms"


def download_agg_trades(
    symbol: str,
    date: str,
    venue: str,
) -> tuple[pd.DataFrame, dict[str, object]]:
    if venue not in {"spot", "perp"}:
        raise ValueError("venue must be 'spot' or 'perp'")

    base = PERP_URL if venue == "perp" else SPOT_URL
    url = f"{base}/{symbol}/{symbol}-aggTrades-{date}.zip"
    raw_path = CACHE_DIR / "raw" / f"{symbol}_{date}_{venue}_aggtrades.zip"
    archive = fetch_archive(url, raw_path, timeout=120.0)

    parsed_path = (
        CACHE_DIR
        / "parsed"
        / f"{symbol}_{date}_{venue}_{archive.sha256[:12]}.pkl"
    )
    if parsed_path.exists():
        frame = pd.read_pickle(parsed_path)
        member = "cached-parsed-frame"
    else:
        names = PERP_COLUMNS if venue == "perp" else SPOT_COLUMNS
        with zipfile.ZipFile(raw_path) as zipped:
            member = zipped.namelist()[0]
            with zipped.open(member) as handle:
                preview = handle.read(300).decode("utf-8", errors="ignore")
                handle.seek(0)
                has_header = "agg_trade_id" in preview or "transact_time" in preview
                frame = pd.read_csv(
                    handle,
                    header=0 if has_header else None,
                    names=None if has_header else names,
                    low_memory=False,
                )

        if "transact_time" not in frame.columns:
            frame.columns = names[: len(frame.columns)]
        for column in ("price", "quantity", "transact_time"):
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(
            subset=["price", "quantity", "transact_time"]
        ).copy()
        unit = _timestamp_unit(frame["transact_time"])
        frame["timestamp"] = pd.to_datetime(
            frame["transact_time"], unit=unit, utc=True
        )
        frame = frame.sort_values("timestamp").reset_index(drop=True)
        parsed_path.parent.mkdir(parents=True, exist_ok=True)
        frame.to_pickle(parsed_path)

    unit = _timestamp_unit(frame["transact_time"])
    metadata = archive.to_dict() | {
        "venue": venue,
        "archive_member": member,
        "parsed_cache": str(parsed_path),
        "rows": int(len(frame)),
        "timestamp_unit": unit,
        "first_timestamp": frame["timestamp"].min(),
        "last_timestamp": frame["timestamp"].max(),
    }
    return frame, metadata


def to_second_bars(
    trades: pd.DataFrame,
    date: str,
    bar_seconds: int,
) -> pd.DataFrame:
    if bar_seconds < 1:
        raise ValueError("BAR_SECONDS must be at least one second")

    rule = f"{bar_seconds}s"
    indexed = trades.set_index("timestamp").sort_index()
    bars = indexed.resample(rule).agg(
        last=("price", "last"),
        volume=("quantity", "sum"),
        trades=("price", "size"),
    )

    day_start = pd.Timestamp(date, tz="UTC")
    day_end = day_start + pd.Timedelta(days=1)
    full_index = pd.date_range(
        day_start,
        day_end - pd.Timedelta(seconds=bar_seconds),
        freq=rule,
    )
    bars = bars.reindex(full_index)
    bars.index.name = "timestamp"

    had_trade = bars["last"].notna()
    last_event_time = pd.Series(
        bars.index.where(had_trade), index=bars.index
    ).ffill()
    bars["age_s"] = (
        bars.index.to_series(index=bars.index) - last_event_time
    ).dt.total_seconds()
    bars["last"] = bars["last"].ffill()
    bars["volume"] = bars["volume"].fillna(0.0)
    bars["trades"] = bars["trades"].fillna(0).astype(int)
    bars["had_trade"] = had_trade
    return bars


perp_trades, perp_archive = download_agg_trades(SYMBOL, DATE, "perp")
spot_trades, spot_archive = download_agg_trades(SYMBOL, DATE, "spot")
perp = to_second_bars(perp_trades, DATE, BAR_SECONDS).add_prefix("perp_")
spot = to_second_bars(spot_trades, DATE, BAR_SECONDS).add_prefix("spot_")

panel = perp.join(spot, how="inner")
panel["valid"] = (
    panel["perp_last"].notna()
    & panel["spot_last"].notna()
    & (panel["perp_age_s"] <= MAX_STALE_SECONDS)
    & (panel["spot_age_s"] <= MAX_STALE_SECONDS)
)
panel["basis_log"] = np.where(
    panel["valid"],
    np.log(panel["perp_last"]) - np.log(panel["spot_last"]),
    np.nan,
)
panel["basis_bp"] = panel["basis_log"] * 1e4
valid_panel = panel.loc[panel["valid"]]

quality = pd.DataFrame({
    "value": {
        "perpetual aggTrades": len(perp_trades),
        "spot aggTrades": len(spot_trades),
        "all one-second timestamps": len(panel),
        "valid aligned timestamps": int(panel["valid"].sum()),
        "valid share": panel["valid"].mean(),
        "basis mean (bp)": valid_panel["basis_bp"].mean(),
        "basis std (bp)": valid_panel["basis_bp"].std(),
        "basis min (bp)": valid_panel["basis_bp"].min(),
        "basis max (bp)": valid_panel["basis_bp"].max(),
        "max perp age retained (s)": valid_panel["perp_age_s"].max(),
        "max spot age retained (s)": valid_panel["spot_age_s"].max(),
    }
})
display(quality)


## 4. Data-quality inspection

The price and basis plots are diagnostic, not trading evidence. The trade-age panel verifies how often a forward-filled last trade approaches the declared staleness boundary.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(panel.index, panel["perp_last"], lw=0.55, label="Perpetual last")
axes[0].plot(panel.index, panel["spot_last"], lw=0.55, alpha=0.8, label="Spot last")
axes[0].set_ylabel("USDT")
axes[0].set_title(f"{SYMBOL} real Binance trades — {DATE}")
axes[0].legend(loc="upper right")

axes[1].plot(panel.index, panel["basis_bp"], lw=0.55)
axes[1].axhline(valid_panel["basis_bp"].mean(), ls="--", lw=1, label="daily mean")
axes[1].set_ylabel("Basis (bp)")
axes[1].legend(loc="upper right")

axes[2].plot(panel.index, panel["perp_age_s"], lw=0.45, label="Perp age")
axes[2].plot(panel.index, panel["spot_age_s"], lw=0.45, alpha=0.8, label="Spot age")
axes[2].axhline(MAX_STALE_SECONDS, ls="--", lw=1)
axes[2].set_ylabel("Age (s)")
axes[2].set_xlabel("UTC")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()


## 5. One-day chronological partitions

The threshold-development interval is isolated from the final holdout. The purge is longer than the maximum configured episode plus fill delays, so a development-period position cannot leak a realised exit into the holdout.


In [ ]:
split = make_intraday_research_split(
    panel.index,
    train_duration=TRAIN_DURATION,
    development_duration=DEVELOPMENT_DURATION,
    purge_duration=pd.Timedelta(seconds=PURGE_SECONDS),
    minimum_holdout_duration=MINIMUM_HOLDOUT_DURATION,
)
display(split.summary())

panel["sample"] = ""
panel.iloc[split.train_positions, panel.columns.get_loc("sample")] = "training"
panel.iloc[
    split.development_positions,
    panel.columns.get_loc("sample"),
] = "threshold_development"
panel.iloc[split.purge_positions, panel.columns.get_loc("sample")] = "purge"
panel.iloc[split.holdout_positions, panel.columns.get_loc("sample")] = "final_holdout"


## 6. Fit the state-space model and causal baselines

The state-space model and AR(1)/OU baseline see only the training interval. The rolling-z comparator uses a reference window ending at `t-1`, so the current observation is excluded from its own normalisation.

The AR(1)/OU baseline is intentionally simple. If the training data cannot support a stationary positive-persistence AR(1), the baseline is marked unavailable rather than coerced into a misleading result.


In [ ]:
y = panel["basis_log"].to_numpy(dtype=float)
train_y = y[split.train_positions]
if np.isfinite(train_y).sum() < 500:
    raise ValueError("Training interval contains fewer than 500 valid observations")

fit = fit_state_space(
    train_y,
    dt=float(BAR_SECONDS),
    max_obs=None,
    burn_in=50,
    compare_null=True,
    require_convergence=True,
)
filtered = filter_state_space(y, fit)

panel["level_bp"] = filtered.level * 1e4
panel["transient_bp"] = filtered.transient * 1e4
panel["innovation_z"] = filtered.standardized_innovation
panel["signal_state_space_z"] = filtered.transient_stationary_z(fit.params)
panel["signal_filter_t"] = filtered.transient_filter_t
panel["signal_rolling_z"] = causal_rolling_z(
    y,
    window=ROLLING_WINDOW_SECONDS // BAR_SECONDS,
    min_periods=ROLLING_MIN_PERIODS // BAR_SECONDS,
)

ar1_params = None
ar1_error = None
try:
    ar1_params = fit_ar1_ou_baseline(
        train_y,
        dt_seconds=float(BAR_SECONDS),
        min_pairs=500,
    )
    panel["signal_ar1_ou_z"] = ar1_ou_signal_z(y, ar1_params)
except ValueError as exc:
    ar1_error = str(exc)
    panel["signal_ar1_ou_z"] = np.nan

holdout_start_position = int(split.holdout_positions[0])
holdout_stop_position = int(split.holdout_positions[-1]) + 1
holdout_filtered = filtered.slice(holdout_start_position, holdout_stop_position)
adequacy = holdout_filtered.adequacy_report(
    max_lag=30,
    alpha=0.01,
    require_gaussian=False,
)

if not adequacy.dynamic_passed and not RUN_DIAGNOSTIC_BACKTEST:
    raise RuntimeError(
        "Final-holdout dynamic adequacy failed and diagnostic backtesting is disabled"
    )


## 7. Model fit, identifiability and final-holdout adequacy

A better BIC than the local-level null is necessary but not sufficient. The final holdout innovations are checked for centring, scale and residual autocorrelation. Gaussianity is reported separately because heavy tails can invalidate Gaussian inference without necessarily invalidating the conditional mean.


In [ ]:
print(fit.report(scale=1e4, unit="bp"))
print()
print(adequacy.summary())
display(pd.Series(adequacy.diagnostics, name="value").to_frame())

if ar1_params is not None:
    display(pd.Series({
        "phi": ar1_params.phi,
        "half-life (s)": ar1_params.half_life_seconds,
        "unconditional mean (bp)": ar1_params.unconditional_mean * 1e4,
        "stationary sd (bp)": ar1_params.stationary_sd * 1e4,
        "training pairs": ar1_params.n_pairs,
    }, name="AR(1)/OU baseline").to_frame())
else:
    print(f"AR(1)/OU baseline unavailable: {ar1_error}")


## 8. One-sided decomposition around the final holdout

`signal_state_space_z` measures economic amplitude relative to the structural stationary transient scale. `signal_filter_t` measures posterior state-estimation confidence. They are plotted together but never used interchangeably.


In [ ]:
view = panel.loc[
    split.holdout_start - pd.Timedelta(hours=1):
    split.holdout_start + pd.Timedelta(hours=3)
]
transient_sd_bp = fit.params.transient_sd * 1e4

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
axes[0].plot(view.index, view["basis_bp"], lw=0.6, label="Observed basis")
axes[0].plot(view.index, view["level_bp"], lw=1.0, label="Filtered level")
axes[0].axvline(split.holdout_start, ls="--", lw=1.2, label="Final holdout")
axes[0].set_ylabel("bp")
axes[0].set_title("Real basis and one-sided state-space decomposition")
axes[0].legend(loc="upper right")

axes[1].plot(view.index, view["transient_bp"], lw=0.65)
axes[1].axhline(ENTRY_Z * transient_sd_bp, ls="--", lw=1, label="+ entry")
axes[1].axhline(-ENTRY_Z * transient_sd_bp, ls="--", lw=1, label="- entry")
axes[1].axhline(0, lw=0.8)
axes[1].set_ylabel("Transient (bp)")
axes[1].legend(loc="upper right")

axes[2].plot(view.index, view["signal_state_space_z"], lw=0.65, label="stationary z")
axes[2].plot(view.index, view["signal_filter_t"], lw=0.45, alpha=0.5, label="filter t")
axes[2].axhline(ENTRY_Z, ls="--", lw=1)
axes[2].axhline(-ENTRY_Z, ls="--", lw=1)
axes[2].axhspan(-EXIT_Z, EXIT_Z, alpha=0.15, label="exit region")
axes[2].axvline(split.holdout_start, ls="--", lw=1.2)
axes[2].set_ylabel("Signal scale")
axes[2].set_xlabel("UTC")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()


## 9. Timestamp-correct final-holdout proxy backtest

The primary configuration is evaluated once on the final holdout.

- Decision and fill timestamps are distinct.
- A fill cannot occur on the decision bar.
- Stale pending actions are cancelled across excessive data gaps.
- Mean-reversion exit is directional: crossing through zero and overshooting the opposite side still exits.
- Closed trades and the end-of-sample state are reported separately. An unfinished position is never silently dropped from the result.
- The open-position mark is hypothetical and is not added to realised P&L.


In [ ]:
backtest_config = ProxyBacktestConfig(
    entry_z=ENTRY_Z,
    exit_z=EXIT_Z,
    stop_z=STOP_Z,
    max_hold_seconds=MAX_HOLD_SECONDS,
    cooldown_seconds=COOLDOWN_SECONDS,
    round_trip_cost_bp=ROUND_TRIP_COST_BP,
    execution_delay_seconds=EXECUTION_DELAY_SECONDS,
    max_fill_delay_seconds=MAX_FILL_DELAY_SECONDS,
    allow_short_spot=ALLOW_SHORT_SPOT,
)

state_space_result = run_proxy_backtest_detailed(
    panel,
    start_time=split.holdout_start,
    end_time=split.holdout_end_exclusive,
    config=backtest_config,
    basis_col="basis_bp",
    signal_col="signal_state_space_z",
    tradable_col="valid",
)
trades = state_space_result.trades

print(f"Round-trip modelled cost: {ROUND_TRIP_COST_BP:.2f} bp")
print(f"Final-holdout clock observations: {split.holdout_positions.size:,}")
print(f"Completed diagnostic trades: {len(trades):,}")
display(trades.head(10))
display(state_space_result.final_state.to_frame())


## 10. Realised proxy economics

Performance statistics below use completed trades only. The final-state table above must be read alongside them whenever an open position or pending instruction remains at the interval boundary.


In [ ]:
def performance_row(
    result: ProxyBacktestResult,
    *,
    model: str,
) -> dict[str, object]:
    closed = result.trades
    row: dict[str, object] = {
        "model": model,
        "trades": len(closed),
        "open_position": result.final_state.has_open_position,
        "pending_action": result.final_state.pending_action,
        "open_mtm_net_if_liquidated_bp": (
            result.final_state.mark_to_market_net_bp_if_liquidated
        ),
        "cancelled_stale_actions": result.final_state.cancelled_stale_actions,
    }
    if closed.empty:
        row.update({
            "gross_total_bp": 0.0,
            "net_total_bp": 0.0,
            "avg_gross_bp": np.nan,
            "avg_net_bp": np.nan,
            "gross_win_rate": np.nan,
            "net_win_rate": np.nan,
            "median_holding_s": np.nan,
            "max_drawdown_bp": np.nan,
        })
        return row

    net_equity = closed["net_pnl_bp"].cumsum()
    drawdown = net_equity - net_equity.cummax()
    row.update({
        "gross_total_bp": closed["gross_pnl_bp"].sum(),
        "net_total_bp": closed["net_pnl_bp"].sum(),
        "avg_gross_bp": closed["gross_pnl_bp"].mean(),
        "avg_net_bp": closed["net_pnl_bp"].mean(),
        "gross_win_rate": (closed["gross_pnl_bp"] > 0).mean(),
        "net_win_rate": (closed["net_pnl_bp"] > 0).mean(),
        "median_holding_s": closed["holding_s"].median(),
        "max_drawdown_bp": drawdown.min(),
    })
    return row


state_space_performance = pd.DataFrame([
    performance_row(state_space_result, model="state_space")
]).set_index("model")
display(state_space_performance.T)

if not trades.empty:
    plot_trades = trades.copy()
    plot_trades["gross_equity_bp"] = plot_trades["gross_pnl_bp"].cumsum()
    plot_trades["net_equity_bp"] = plot_trades["net_pnl_bp"].cumsum()

    fig, axes = plt.subplots(3, 1, figsize=(13, 10))
    axes[0].plot(plot_trades["exit_time"], plot_trades["gross_equity_bp"], label="Gross")
    axes[0].plot(plot_trades["exit_time"], plot_trades["net_equity_bp"], label="Net proxy")
    axes[0].axhline(0, lw=0.8)
    axes[0].set_ylabel("Cumulative bp")
    axes[0].set_title("Final-holdout proxy equity")
    axes[0].legend()

    axes[1].hist(plot_trades["gross_pnl_bp"], bins=40, alpha=0.65, label="Gross")
    axes[1].hist(plot_trades["net_pnl_bp"], bins=40, alpha=0.65, label="Net")
    axes[1].axvline(0, lw=0.8)
    axes[1].set_xlabel("Trade P&L (bp)")
    axes[1].set_ylabel("Count")
    axes[1].legend()

    axes[2].scatter(
        plot_trades["entry_signal_z"],
        plot_trades["gross_pnl_bp"],
        s=15,
        alpha=0.55,
    )
    axes[2].axhline(0, lw=0.8)
    axes[2].set_xlabel("Entry transient z")
    axes[2].set_ylabel("Gross P&L (bp)")
    axes[2].set_title("Signal amplitude versus gross convergence")
    plt.tight_layout()
    plt.show()


## 11. Final-holdout baseline comparison

All available models use the same predeclared threshold, tradability mask, fill delay, holding rules and costs. This isolates the incremental value of the state-space signal from execution-rule differences.

The rolling-z baseline is causal. The AR(1)/OU baseline is fitted only on training data. An explicit no-trade row anchors interpretation at zero activity and zero P&L. No baseline is selected or tuned using final-holdout performance.


In [ ]:
model_signals = {
    "state_space": "signal_state_space_z",
    "causal_rolling_z": "signal_rolling_z",
}
if ar1_params is not None:
    model_signals["training_ar1_ou"] = "signal_ar1_ou_z"

holdout_results: dict[str, ProxyBacktestResult] = {}
for model_name, signal_column in model_signals.items():
    holdout_results[model_name] = run_proxy_backtest_detailed(
        panel,
        start_time=split.holdout_start,
        end_time=split.holdout_end_exclusive,
        config=backtest_config,
        basis_col="basis_bp",
        signal_col=signal_column,
        tradable_col="valid",
    )

comparison_rows = [
    performance_row(result, model=model_name)
    for model_name, result in holdout_results.items()
]
comparison_rows.append({
    "model": "no_trade",
    "trades": 0,
    "open_position": False,
    "pending_action": None,
    "open_mtm_net_if_liquidated_bp": None,
    "cancelled_stale_actions": 0,
    "gross_total_bp": 0.0,
    "net_total_bp": 0.0,
    "avg_gross_bp": np.nan,
    "avg_net_bp": np.nan,
    "gross_win_rate": np.nan,
    "net_win_rate": np.nan,
    "median_holding_s": np.nan,
    "max_drawdown_bp": 0.0,
})
comparison = pd.DataFrame(comparison_rows).set_index("model")
display(comparison.round(4))


## 12. Exploratory threshold sensitivity with FDR control

This sweep runs **only in the threshold-development interval**. It does not inspect the final holdout and it does not replace the predeclared `ENTRY_Z=2.0` result.

For thresholds with enough completed episodes, a centred-null circular block-bootstrap tests whether mean gross convergence is positive. Benjamini–Hochberg adjusted values are reported across the exploratory family. These are dependent-data diagnostics, not a licence to promote the best-looking threshold.


In [ ]:
sweep_rows: list[dict[str, object]] = []
for threshold in THRESHOLD_GRID:
    candidate_config = ProxyBacktestConfig(
        entry_z=float(threshold),
        exit_z=EXIT_Z,
        stop_z=max(STOP_Z, float(threshold) + 1.5),
        max_hold_seconds=MAX_HOLD_SECONDS,
        cooldown_seconds=COOLDOWN_SECONDS,
        round_trip_cost_bp=ROUND_TRIP_COST_BP,
        execution_delay_seconds=EXECUTION_DELAY_SECONDS,
        max_fill_delay_seconds=MAX_FILL_DELAY_SECONDS,
        allow_short_spot=ALLOW_SHORT_SPOT,
    )
    candidate = run_proxy_backtest_detailed(
        panel,
        start_time=split.development_start,
        end_time=split.development_end_exclusive,
        config=candidate_config,
        basis_col="basis_bp",
        signal_col="signal_state_space_z",
        tradable_col="valid",
    )
    closed = candidate.trades
    p_value = np.nan
    if len(closed) >= 10:
        block_length = max(2, round(len(closed) ** (1 / 3)))
        test = circular_block_bootstrap_mean_test(
            closed["gross_pnl_bp"].to_numpy(),
            block_length=block_length,
            n_boot=BOOTSTRAP_REPLICATIONS,
            alternative="greater",
            seed=17,
        )
        p_value = float(test["p_value"])

    sweep_rows.append({
        "entry_z": float(threshold),
        "trades": len(closed),
        "gross_total_bp": 0.0 if closed.empty else closed["gross_pnl_bp"].sum(),
        "net_total_bp": 0.0 if closed.empty else closed["net_pnl_bp"].sum(),
        "avg_gross_bp": np.nan if closed.empty else closed["gross_pnl_bp"].mean(),
        "avg_net_bp": np.nan if closed.empty else closed["net_pnl_bp"].mean(),
        "gross_mean_p": p_value,
        "open_at_end": candidate.final_state.has_open_position,
    })

sweep = pd.DataFrame(sweep_rows)
sweep["gross_mean_fdr_q"] = np.nan
finite_p = sweep["gross_mean_p"].notna()
if finite_p.any():
    sweep.loc[finite_p, "gross_mean_fdr_q"] = benjamini_hochberg(
        sweep.loc[finite_p, "gross_mean_p"].to_numpy()
    )
display(sweep.round(5))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(sweep["entry_z"], sweep["trades"], marker="o")
axes[0].set_ylabel("Completed trades")
axes[0].set_title("Development-only threshold sensitivity")
axes[1].plot(sweep["entry_z"], sweep["gross_total_bp"], marker="o", label="Gross")
axes[1].plot(sweep["entry_z"], sweep["net_total_bp"], marker="o", label="Net proxy")
axes[1].axhline(0, lw=0.8)
axes[1].set_xlabel("Entry threshold (stationary z)")
axes[1].set_ylabel("Total P&L (bp)")
axes[1].legend()
plt.tight_layout()
plt.show()


## 13. Final economic and statistical decision

The decision table combines model preference, identifiability, final-holdout adequacy, signal scale, explicit costs and dependent-data uncertainty. A failure at any required gate keeps the result at diagnostic status.

The bootstrap is applied to completed final-holdout episodes only. An open terminal position remains separate in the final-state report.


In [ ]:
cost_to_signal = ROUND_TRIP_COST_BP / transient_sd_bp
bic_improvement = fit.delta_bic_vs_local_level
identification = fit.identifiability_summary()

bootstrap = None
if len(trades) >= 20:
    bootstrap = circular_block_bootstrap_mean_ci(
        trades["gross_pnl_bp"].to_numpy(),
        block_length=max(2, round(len(trades) ** (1 / 3))),
        n_boot=BOOTSTRAP_REPLICATIONS,
        confidence=0.95,
        seed=7,
    )

decision: dict[str, object] = {
    "scope": "single settled UTC day by design",
    "real data": True,
    "training clock observations": split.train_positions.size,
    "development clock observations": split.development_positions.size,
    "purge clock observations": split.purge_positions.size,
    "final-holdout clock observations": split.holdout_positions.size,
    "training valid observations": int(np.isfinite(train_y).sum()),
    "half-life (s)": fit.half_life,
    "transient stationary sd (bp)": transient_sd_bp,
    "round-trip modelled cost (bp)": ROUND_TRIP_COST_BP,
    "cost / transient sd": cost_to_signal,
    "BIC improvement vs local-level null": bic_improvement,
    "near-optimal half-life ratio": identification["half_life_ratio"],
    "near-optimal signal-scale ratio": identification["transient_sd_ratio"],
    "final-holdout dynamic adequacy passed": adequacy.dynamic_passed,
    "final-holdout Gaussian innovations passed": adequacy.gaussian_passed,
    "completed final-holdout trades": len(trades),
    "open final position": state_space_result.final_state.has_open_position,
    "open MTM net if liquidated (bp)": (
        state_space_result.final_state.mark_to_market_net_bp_if_liquidated
    ),
    "gross completed-trade total (bp)": (
        0.0 if trades.empty else trades["gross_pnl_bp"].sum()
    ),
    "net completed-trade total (bp)": (
        0.0 if trades.empty else trades["net_pnl_bp"].sum()
    ),
}
if bootstrap is not None:
    decision.update({
        "gross mean bootstrap lower (bp/trade)": bootstrap["lower"],
        "gross mean bootstrap upper (bp/trade)": bootstrap["upper"],
        "bootstrap P(mean > 0)": bootstrap["probability_mean_positive"],
    })
display(pd.Series(decision, name="value").to_frame())

print("INTERPRETATION")
if bic_improvement is not None and bic_improvement > 10:
    print("- The transient model is strongly preferred to the local-level null by BIC.")
else:
    print("- The transient model is not strongly preferred to the local-level null.")
if not adequacy.dynamic_passed:
    print("- Final-holdout innovations fail the dynamic adequacy gate.")
if not adequacy.gaussian_passed:
    print("- Gaussian innovations are rejected; robust inference is required.")
if cost_to_signal > 3:
    print("- Modelled round-trip cost exceeds three transient standard deviations.")
else:
    print("- Cost is within three transient standard deviations; this only warrants execution research.")
print("- Closed-trade P&L and terminal open inventory are reported separately.")
print("- No executable-fill claim is made from aggTrades data.")


## 14. Persist the fit and run manifest

The manifest records the exact UTC day, archive URLs and SHA-256 digests, parsed row counts, configuration, package versions, partition boundaries and principal outputs. This makes a reproduced run auditable even when a remote archive or local environment later changes.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fit_path = OUTPUT_DIR / f"{SYMBOL}_{DATE}_state_space_fit.json"
manifest_path = OUTPUT_DIR / f"{SYMBOL}_{DATE}_research_manifest.json"
fit.save_json(fit_path)

manifest = write_research_manifest(
    manifest_path,
    study={
        "name": "BTC spot-perpetual single-day causal state-space research",
        "scope": "single-day by design; multi-day research is a separate project",
        "symbol": SYMBOL,
        "utc_date": DATE,
        "bar_seconds": BAR_SECONDS,
        "training_start": split.train_start,
        "training_end_exclusive": split.train_end_exclusive,
        "development_start": split.development_start,
        "development_end_exclusive": split.development_end_exclusive,
        "holdout_start": split.holdout_start,
        "holdout_end_exclusive": split.holdout_end_exclusive,
    },
    archives=[perp_archive, spot_archive],
    configuration={
        "max_stale_seconds": MAX_STALE_SECONDS,
        "entry_z": ENTRY_Z,
        "exit_z": EXIT_Z,
        "stop_z": STOP_Z,
        "max_hold_seconds": MAX_HOLD_SECONDS,
        "cooldown_seconds": COOLDOWN_SECONDS,
        "execution_delay_seconds": EXECUTION_DELAY_SECONDS,
        "max_fill_delay_seconds": MAX_FILL_DELAY_SECONDS,
        "round_trip_cost_bp": ROUND_TRIP_COST_BP,
        "allow_short_spot": ALLOW_SHORT_SPOT,
        "rolling_window_seconds": ROLLING_WINDOW_SECONDS,
    },
    outputs={
        "fit_json": str(fit_path),
        "completed_holdout_trades": len(trades),
        "terminal_state": state_space_result.final_state.to_frame()["value"].to_dict(),
        "dynamic_adequacy_passed": adequacy.dynamic_passed,
        "gaussian_adequacy_passed": adequacy.gaussian_passed,
        "delta_bic_vs_local_level": bic_improvement,
    },
    package_versions=installed_versions([
        "numpy", "scipy", "numba", "pandas", "matplotlib", "ipython"
    ]),
)
print(f"Wrote fit: {fit_path}")
print(f"Wrote manifest: {manifest}")


## 15. Reusable signal semantics

### Historical fit

```python
from state_space import fit_state_space, filter_state_space

fit = fit_state_space(training_basis_log, dt=1.0)
result = filter_state_space(all_basis_log, fit)

signal_stationary_z = result.transient_stationary_z(fit.params)
signal_filter_t = result.transient_filter_t
```

### Online update

```python
from state_space import OnlineBasisFilter

online = OnlineBasisFilter.from_history(fit.params, training_basis_log)
state = online.update(new_basis_log)
print(state["transient_stationary_z"], state["transient_filter_t"])
```

The state-space module estimates and filters the signal. Execution, inventory, funding, borrow, order acknowledgement, partial fills and reconciliation remain separate systems.


## 16. Boundary and handoff

This notebook ends at a trade-price proxy and an explicit economic gate.

A separate execution-research project would require synchronized bid/ask or depth data, crossing-versus-maker assumptions, two-leg latency, partial fills, cancellation state, adverse-selection measurement and the account's own order/execution telemetry.

Likewise, a separate multi-day project would address cross-date refits, regime stability and prospective evaluation. Neither extension should be silently folded into this deliberately single-day repository.
